# Análise Exploratória — Observatório Macro-Financeiro

Notebook de exploração das séries oficiais do SGS/BCB (2020–2023).

**Antes da entrega:** execute todas as células com acesso à internet e salve o notebook com os resultados e gráficos visíveis. Os resultados não são pré-preenchidos para evitar apresentar números não verificados.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.etl import run_etl, validate_data

df = run_etl()
df.head()

## 1. Estatísticas descritivas e qualidade

In [ ]:
display(df.describe().T)
diagnostico = validate_data(df)
diagnostico

## 2. Tratamento de nulos e duplicidades

In [ ]:
print("Nulos por coluna:")
display(df.isna().sum().to_frame("quantidade"))
print("Duplicidades:", df.duplicated().sum())
df_limpo = df.drop_duplicates(subset=["date"]).sort_values("date").copy()
df_limpo["date"] = pd.to_datetime(df_limpo["date"], errors="coerce")
for coluna in df_limpo.columns:
    if coluna != "date":
        df_limpo[coluna] = pd.to_numeric(df_limpo[coluna], errors="coerce")
df_limpo = df_limpo.dropna(subset=["date"])
display(df_limpo.isna().sum().to_frame("nulos após padronização"))

## 3. Detecção exploratória de outliers (IQR)

In [ ]:
numericas = df_limpo.select_dtypes(include="number")
resumo_outliers = []
for coluna in numericas.columns:
    q1, q3 = numericas[coluna].quantile([0.25, 0.75])
    iqr = q3 - q1
    inferior, superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    qtd = ((numericas[coluna] < inferior) | (numericas[coluna] > superior)).sum()
    resumo_outliers.append({"variavel": coluna, "limite_inferior": inferior,
                            "limite_superior": superior, "outliers_iqr": int(qtd)})
pd.DataFrame(resumo_outliers)

## 4. Distribuições e dispersão

In [ ]:
colunas = ["selic_meta", "ipca_12m", "dolar_ptax", "ibc_br"]
for coluna in colunas:
    ax = df_limpo[coluna].dropna().plot(kind="hist", bins=15, title=f"Distribuição — {coluna}")
    ax.set_xlabel(coluna)
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df_limpo["selic_meta"], df_limpo["ipca_12m"], alpha=0.7)
ax.set_title("Dispersão: Selic Meta x IPCA 12 meses")
ax.set_xlabel("Selic Meta (%)")
ax.set_ylabel("IPCA 12 meses (%)")
plt.show()

## 5. Correlação

In [ ]:
corr = df_limpo[["selic_meta", "ipca_mensal", "ipca_12m",
                    "dolar_ptax", "ibc_br"]].corr(method="pearson")
display(corr)
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr.index)), corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
fig.colorbar(im, ax=ax)
ax.set_title("Matriz de correlação de Pearson")
plt.tight_layout()
plt.show()

## 6. Conclusão da EDA

Após executar as células, escreva aqui três observações apoiadas nos resultados gerados: (1) comportamento da Selic e inflação; (2) trajetória do dólar e IBC-Br; (3) correlações mais altas. Não interprete correlação como causalidade.